# 05 · LLM Decoder-only (GPT-style) pretrained: zero-shot, few-shot, fine-tune LoRA

Mô hình: **Qwen2.5-1.5B-Instruct** (hỗ trợ tiếng Việt, có chat template). Ba thí nghiệm:
1. **Zero-shot**: chỉ dùng câu lệnh (prompt) yêu cầu dịch.
2. **Few-shot (5-shot)**: thêm 5 cặp câu mẫu vào hội thoại (in-context learning).
3. **Fine-tune LoRA**: đóng băng mô hình gốc, chỉ học ma trận hạng thấp (≈1% tham số); loss chỉ tính trên câu trả lời.

**Các điểm kỹ thuật đã xử lý:** đệm trái khi sinh theo batch (bắt buộc với decoder-only); hậu xử lý đầu ra
(LLM hay thêm "Bản dịch:", dấu ngoặc kép, lời giải thích); kiểm tra chat template nhất quán giữa lúc train và lúc dịch;
trên GPU < 30GB tự bật gradient checkpointing; checkpoint chỉ lưu tham số LoRA (vài chục MB).

⏱️ Ước tính (A100): zero-shot/few-shot ~5–10 phút mỗi loại; LoRA 1 epoch toàn bộ train ~1–2 giờ (T4: dùng `TRAIN_SUBSET`).

> **Quy trình chống lỗi:** luôn chạy với `DRY_RUN = True` trước (vài phút, dữ liệu và mô hình tí hon, đi qua **mọi**
> dòng code). Khi không còn lỗi nào, đổi `DRY_RUN = False` rồi chọn **Runtime → Run all** để chạy thật.
> Kết quả chạy thử nằm riêng trong thư mục `dryrun/`, không lẫn với kết quả thật.

> **Nếu Colab mất kết nối / hết phiên:** mở lại notebook, **Run all**. Huấn luyện tự tiếp tục từ checkpoint gần nhất
> (lưu mỗi vài trăm bước trên Drive). Bản dịch đã sinh từ đúng checkpoint sẽ không bị dịch lại.

In [1]:
#@title ⚙️ Tham số
DRY_RUN = False
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"     # có thể đổi: Qwen/Qwen2.5-0.5B-Instruct, Qwen/Qwen2.5-3B-Instruct
DRY_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct" # mô hình nhỏ dùng khi DRY_RUN
RUN_ZERO_SHOT = True
RUN_FEW_SHOT = True
N_SHOTS = 5
RUN_LORA = True
TRAIN_SUBSET = None        # None = toàn bộ train; vd 40000 để nhanh hơn (T4)
LORA = dict(r=16, lora_alpha=32, lora_dropout=0.05,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
FT = dict(epochs=1, lr=2e-4, warmup_steps=100, schedule="cosine", max_tokens=8192, grad_accum=1,
          eval_every_steps=500, save_every_steps=200, patience=4)
DEV_EVAL_SIZE = 300
GEN_BATCH = 32
TIME_BUDGET_HOURS = None

In [2]:
#@title 🔧 Kết nối Google Drive + nạp thư viện `envi_lib`
import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
PROJECT_DIR = os.environ.get("ENVI_PROJECT_DIR", "/content/drive/MyDrive/NLP_Final_EnVi")
if not os.path.exists(os.path.join(PROJECT_DIR, "envi_lib", "__init__.py")):
    raise FileNotFoundError(f"Chưa có thư viện ở {PROJECT_DIR}/envi_lib — hãy chạy notebook 00_cai_dat.ipynb trước.")
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
from envi_lib.common import (Paths, set_seed, get_device, pick_amp_dtype, autocast_ctx, gpu_info, peak_gpu_mem_gb,
                             reset_peak_mem, free_memory, count_params, disk_free_gb, save_predictions,
                             prediction_is_current, save_json, load_json, read_lines, write_lines, fmt_seconds,
                             bf16_supported, check_lib_version, now_str)
check_lib_version("1.0.0")
paths = Paths(PROJECT_DIR, dry_run=DRY_RUN).make_all()
DEVICE = get_device()
print(f"📁 {paths.root}   |   DRY_RUN = {DRY_RUN}   |   thiết bị: {gpu_info()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📁 /content/drive/MyDrive/NLP_Final_EnVi   |   DRY_RUN = False   |   thiết bị: {'cuda': True, 'torch': '2.11.0+cu130', 'name': 'NVIDIA L4', 'total_gb': 22.0, 'bf16': True}


In [3]:
#@title 🔄 Cập nhật thư viện (bản sửa lỗi mới nhất) — tự động, không cần chạy lại notebook 00
import sys
_FILES = {
    'hf_utils.py': '"""Làm việc với mô hình pretrained của Hugging Face — tương thích transformers 4.x và 5.x.\n\nNhững điểm khác biệt giữa các phiên bản đã được xử lý tập trung ở đây:\n  * tham số kiểu số thực khi tải mô hình: `dtype` (mới) / `torch_dtype` (cũ)\n  * `batch_decode` gộp vào `decode` ở v5\n  * v5 mặc định use_cache=False => truyền use_cache=True khi generate để sinh nhanh\n  * pipeline \'translation\' bị bỏ ở v5 => không dùng pipeline\n"""\nfrom __future__ import annotations\n\nimport json\nimport os\nimport re\n\nimport torch\n\n# DataLoader dùng tiến trình con (fork); tắt song song của tokenizers để tránh cảnh báo/treo\nos.environ.setdefault("TOKENIZERS_PARALLELISM", "false")\n\n\ndef patch_peft_old_torchao():\n    """Colab cài sẵn torchao bản cũ (0.10); peft >= 0.18 thấy torchao < 0.16 là báo ImportError ngay khi tạo LoRA,\n    dù đồ án KHÔNG dùng torchao (chỉ dùng cho lượng tử hoá).  Ta báo cho peft là "không có torchao".\n    Không gỡ/cài lại gói nào => không ảnh hưởng môi trường Colab."""\n    import importlib\n    import importlib.util\n    import sys\n    if importlib.util.find_spec("torchao") is None or importlib.util.find_spec("peft") is None:\n        return False\n    try:\n        from importlib.metadata import version\n        from packaging.version import Version\n        if Version(version("torchao")) >= Version("0.16.0"):\n            return False\n    except Exception:  # noqa: BLE001\n        return False\n    patched = False\n    for name in ("peft.import_utils", "peft.tuners.lora.torchao", "peft.utils.quantization_utils"):\n        try:\n            mod = sys.modules.get(name) or importlib.import_module(name)\n        except Exception:  # noqa: BLE001\n            continue\n        if hasattr(mod, "is_torchao_available"):\n            mod.is_torchao_available = lambda: False\n            patched = True\n    if patched:\n        print("ℹ️  Phát hiện torchao bản cũ của Colab — đã báo peft bỏ qua torchao (không dùng tới trong đồ án).")\n    return patched\n\n\npatch_peft_old_torchao()\n\nfrom .common import autocast_ctx, normalize_text\nfrom .data import sorted_eval_batches\n\n\ndef tf_version():\n    import transformers\n    from packaging.version import Version\n    return Version(transformers.__version__)\n\n\ndef dtype_kw(dtype):\n    """Tham số kiểu số thực cho from_pretrained theo phiên bản transformers."""\n    from packaging.version import Version\n    return {"dtype": dtype} if tf_version() >= Version("4.56.0") else {"torch_dtype": dtype}\n\n\ndef resolve_model_id(model_id: str) -> str:\n    """Cho phép thay đường dẫn mô hình (dùng khi kiểm thử offline): ENVI_MODEL_OVERRIDES=\'{"id": "path"}\'."""\n    ov = os.environ.get("ENVI_MODEL_OVERRIDES")\n    if ov:\n        return json.loads(ov).get(model_id, model_id)\n    return model_id\n\n\ndef decode_batch(tok, ids):\n    if torch.is_tensor(ids):\n        ids = ids.tolist()\n    if hasattr(tok, "batch_decode"):\n        return tok.batch_decode(ids, skip_special_tokens=True)\n    out = tok.decode(ids, skip_special_tokens=True)\n    return out if isinstance(out, list) else [out]\n\n\n# ----------------------------------------------------------------------------\n# Mô hình Encoder-Decoder pretrained (Marian, T5/envit5, mT5, NLLB)\n# ----------------------------------------------------------------------------\n\nclass Seq2SeqSpec:\n    """Mô tả cách dùng từng họ mô hình: tiền tố đầu vào, ngôn ngữ đích, hậu xử lý đầu ra."""\n\n    def __init__(self, family: str, src_prefix: str = "", tgt_prefix: str = "",\n                 src_lang: str | None = None, tgt_lang: str | None = None, fp16_ok: bool = True):\n        self.family = family\n        self.src_prefix = src_prefix\n        self.tgt_prefix = tgt_prefix          # tiền tố trong NHÃN khi fine-tune (envit5: "vi: ")\n        self.src_lang = src_lang              # NLLB: eng_Latn\n        self.tgt_lang = tgt_lang              # NLLB: vie_Latn\n        self.fp16_ok = fp16_ok                # họ T5 bị tràn số (NaN) với fp16\n\n    def make_src(self, s):\n        return self.src_prefix + s\n\n    def make_tgt(self, t):\n        return self.tgt_prefix + t\n\n    def postprocess(self, out: str) -> str:\n        out = normalize_text(out)\n        if self.tgt_prefix and out.lower().startswith(self.tgt_prefix.strip().lower()):\n            out = out[len(self.tgt_prefix.strip()):].lstrip(" :")\n        return normalize_text(out)\n\n\n# File cần để dựng lại tokenizer từ mô hình SentencePiece gốc (bỏ qua tokenizer.json)\n_TOK_FILES = ["tokenizer_config.json", "special_tokens_map.json", "added_tokens.json", "config.json",\n              "*.model", "*.spm", "vocab*.json", "vocab*.txt", "merges*.txt"]\n\n\ndef _tokenizer_from_sentencepiece(path, **tok_kw):\n    """Dựng tokenizer KHÔNG dùng tokenizer.json, chỉ dùng file SentencePiece gốc (spiece.model ...).\n\n    Dùng khi tokenizer.json của mô hình được tạo bằng phiên bản cũ, không đọc được bằng transformers 5.x\n    (vd. VietAI/envit5-translation: lỗi "\'dict\' object is not an instance of \'Sequence\'").\n    """\n    import shutil\n    import tempfile\n    from pathlib import Path\n    from transformers import AutoTokenizer\n    if os.path.isdir(path):\n        src = Path(path)\n    else:\n        from huggingface_hub import snapshot_download\n        src = Path(snapshot_download(path, allow_patterns=_TOK_FILES))\n    if not any(src.glob("*.model")) and not any(src.glob("*.spm")):\n        raise FileNotFoundError(f"{path}: không có file SentencePiece (*.model) để dựng lại tokenizer")\n    try:\n        import google.protobuf  # noqa: F401  (transformers cần protobuf để đọc file .model)\n    except ImportError as e:\n        raise ImportError("Cần gói protobuf để đọc spiece.model: chạy `%pip install protobuf` rồi "\n                          "Runtime → Restart session, sau đó chạy lại notebook.") from e\n    dst = Path(tempfile.mkdtemp(prefix="tok_"))\n    for f in src.iterdir():\n        if f.is_file() and f.name != "tokenizer.json":\n            shutil.copy(f, dst / f.name)\n    return AutoTokenizer.from_pretrained(str(dst), **tok_kw)\n\n\ndef load_tokenizer(path, **tok_kw):\n    from transformers import AutoTokenizer\n    try:\n        return AutoTokenizer.from_pretrained(path, **tok_kw)\n    except (TypeError, ValueError, KeyError, AttributeError, OSError) as e:\n        if isinstance(e, OSError) and "tokenizer" not in str(e).lower():\n            raise   # lỗi mạng / không tìm thấy mô hình: báo ngay, không che giấu\n        print(f"⚠️  Không đọc được tokenizer.json của {path} ({type(e).__name__}: {str(e)[:120]}).\\n"\n              "   → Dựng lại tokenizer từ file SentencePiece gốc.")\n        return _tokenizer_from_sentencepiece(path, **tok_kw)\n\n\ndef check_tokenizer(tok, model,\n                    sample="Tôi là giáo viên Tin học ở Thành phố Hồ Chí Minh. I teach C++ to students."):\n    """Kiểm tra tokenizer có khớp mô hình không.  Trả về (lỗi_nặng, số_token_<unk>_trên_câu_mẫu).\n\n    Lỗi nặng: id <pad>/</s> khác mô hình; số token vượt xa bảng embedding.\n    """\n    hard = []\n    V = model.get_input_embeddings().weight.shape[0]\n    if len(tok) > 1.01 * V:\n        hard.append(f"tokenizer có {len(tok)} token nhưng bảng embedding chỉ có {V}")\n    cfg = model.config\n    for name in ("pad_token_id", "eos_token_id"):\n        a, b = getattr(tok, name, None), getattr(cfg, name, None)\n        if a is not None and b is not None and not isinstance(b, list) and a != b:\n            hard.append(f"{name}: tokenizer = {a}, mô hình = {b}")\n    ids = tok(sample, add_special_tokens=False)["input_ids"]\n    n_unk = ids.count(tok.unk_token_id) if tok.unk_token_id is not None else 0\n    if len(ids) < len(sample.split()) // 2:\n        hard.append(f"câu mẫu {len(sample.split())} từ chỉ được tách thành {len(ids)} token")\n    return hard, n_unk\n\n\ndef load_seq2seq(model_id, spec: Seq2SeqSpec, device, for_training=False):\n    from transformers import AutoModelForSeq2SeqLM\n    path = resolve_model_id(model_id)\n    tok_kw = {"src_lang": spec.src_lang, "tgt_lang": spec.tgt_lang} if spec.src_lang else {}\n    tok = load_tokenizer(path, **tok_kw)\n    model = AutoModelForSeq2SeqLM.from_pretrained(path, **dtype_kw(torch.float32)).to(device)\n    hard, n_unk = check_tokenizer(tok, model)\n    if hard or n_unk > 2:\n        # tokenizer.json đọc được nhưng có dấu hiệu KHÔNG khớp mô hình => thử dựng lại từ SentencePiece gốc\n        print(f"⚠️  Tokenizer nghi không khớp mô hình ({\'; \'.join(hard) or f\'{n_unk} token <unk> trên câu mẫu\'}).")\n        try:\n            tok2 = _tokenizer_from_sentencepiece(path, **tok_kw)\n            hard2, n_unk2 = check_tokenizer(tok2, model)\n            if not hard2 and (hard or n_unk2 < n_unk):\n                print("   → Dùng tokenizer dựng lại từ file SentencePiece gốc.")\n                tok, hard, n_unk = tok2, hard2, n_unk2\n        except (FileNotFoundError, OSError, ValueError, TypeError) as e:\n            print(f"   (không dựng lại được từ SentencePiece: {str(e)[:100]})")\n    if hard:\n        raise RuntimeError("Tokenizer không khớp mô hình: " + "; ".join(hard))\n    if n_unk > 2:\n        print(f"⚠️  Tokenizer sinh {n_unk} token <unk> trên câu mẫu — kết quả dịch có thể bị ảnh hưởng.")\n    preflight_seq2seq(tok, model, spec)\n    model.train(for_training)\n    return tok, model\n\n\ndef preflight_seq2seq(tok, model, spec: Seq2SeqSpec):\n    """Kiểm tra các yêu cầu đặc thù TRƯỚC khi chạy lâu."""\n    vocab = tok.get_vocab()\n    if spec.family == "marian":\n        m = re.match(r"\\s*(>>\\w+<<)", spec.src_prefix)\n        if m and m.group(1) not in vocab:\n            raise RuntimeError(f"Tokenizer Marian không có token {m.group(1)} — kiểm tra lại mã ngôn ngữ đích.")\n    if spec.tgt_lang:\n        tid = tok.convert_tokens_to_ids(spec.tgt_lang)\n        if tid is None or tid == tok.unk_token_id:\n            raise RuntimeError(f"Không tìm thấy mã ngôn ngữ {spec.tgt_lang} trong tokenizer.")\n    if model.config.decoder_start_token_id is None and getattr(model.generation_config, "decoder_start_token_id", None) is None:\n        raise RuntimeError("Mô hình thiếu decoder_start_token_id.")\n\n\ndef seq2seq_gen_kwargs(tok, spec: Seq2SeqSpec, beam_size, max_new_tokens):\n    # Encoder-decoder: max_length tính theo số token phía decoder (+1 token khởi đầu). Dùng max_length thay vì\n    # max_new_tokens để ghi đè hẳn giá trị mặc định trong generation_config (Marian đặt sẵn 512) mà không bị cảnh báo.\n    kw = dict(num_beams=beam_size, max_length=max_new_tokens + 1, do_sample=False, use_cache=True,\n              length_penalty=1.0, early_stopping=beam_size > 1)\n    if beam_size == 1:\n        kw.pop("early_stopping")\n    if spec.tgt_lang:\n        kw["forced_bos_token_id"] = tok.convert_tokens_to_ids(spec.tgt_lang)\n    return kw\n\n\n@torch.no_grad()\ndef hf_seq2seq_translate(model, tok, spec: Seq2SeqSpec, srcs, beam_size=4, batch_size=32,\n                         max_src_tokens=256, amp_dtype=None, progress=True):\n    from tqdm.auto import tqdm\n    model.eval()\n    device = next(model.parameters()).device\n    texts = [spec.make_src(s) for s in srcs]\n    lens = [len(t) for t in texts]\n    bs = max(1, batch_size // max(1, beam_size // 2))\n    batches = sorted_eval_batches(lens, bs)\n    out = [None] * len(texts)\n    use_amp = amp_dtype if (amp_dtype != torch.float16 or spec.fp16_ok) else None\n    it = tqdm(batches, desc=f"Dịch (beam={beam_size})", leave=False) if progress else batches\n    for idx in it:\n        enc = tok([texts[i] for i in idx], return_tensors="pt", padding=True, truncation=True,\n                  max_length=max_src_tokens)\n        enc = {k: enc[k].to(device) for k in ("input_ids", "attention_mask")}  # bỏ token_type_ids nếu có\n        max_new = int(enc["input_ids"].size(1) * 1.6 + 10)\n        with autocast_ctx(use_amp):\n            gen = model.generate(**enc, **seq2seq_gen_kwargs(tok, spec, beam_size, max_new))\n        for i, h in zip(idx, decode_batch(tok, gen)):\n            out[i] = spec.postprocess(h)\n    return out\n\n\ndef make_hf_seq2seq_features(tok, spec: Seq2SeqSpec, srcs, tgts, max_len=256):\n    """Token hoá cặp câu cho fine-tune.  Trả về list (input_ids, labels) và số cặp bị cắt."""\n    enc = tok([spec.make_src(s) for s in srcs], text_target=[spec.make_tgt(t) for t in tgts],\n              truncation=True, max_length=max_len)\n    feats = list(zip(enc["input_ids"], enc["labels"]))\n    n_trunc = sum(1 for a, b in feats if len(a) >= max_len or len(b) >= max_len)\n    return feats, n_trunc\n\n\ndef hf_seq2seq_collate(batch, pad_id):\n    """batch: list (idx, input_ids, labels)."""\n    from .data import IGNORE, _pad\n    ids = _pad([b[1] for b in batch], pad_id)\n    lab = _pad([b[2] for b in batch], IGNORE)\n    return {"input_ids": ids, "attention_mask": (ids != pad_id) if pad_id is not None else torch.ones_like(ids, dtype=torch.bool),\n            "labels": lab}\n\n\ndef hf_seq2seq_loss(model, batch, cfg):\n    """Tự tạo decoder_input_ids từ nhãn và tự tính loss (có label smoothing).\n\n    Không truyền `labels` vào mô hình để tránh HF tính thêm một lần cross-entropy trên toàn bộ\n    từ vựng (mT5/NLLB có 250k từ => tốn thêm vài GB bộ nhớ GPU).\n    """\n    from .models import label_smoothed_ce\n    labels = batch["labels"]\n    base = model.get_base_model() if hasattr(model, "get_base_model") else model\n    if hasattr(base, "prepare_decoder_input_ids_from_labels"):\n        dec_in = base.prepare_decoder_input_ids_from_labels(labels=labels)\n        out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"].long(),\n                    decoder_input_ids=dec_in)\n    else:\n        out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"].long(), labels=labels)\n    return label_smoothed_ce(out.logits, labels, cfg.label_smoothing)\n\n\nclass FeatDataset(torch.utils.data.Dataset):\n    def __init__(self, feats):\n        self.feats = feats\n\n    def __len__(self):\n        return len(self.feats)\n\n    def __getitem__(self, i):\n        a, b = self.feats[i]\n        return i, a, b\n\n\n# ----------------------------------------------------------------------------\n# LLM decoder-only (Qwen ...) — prompt, LoRA\n# ----------------------------------------------------------------------------\n\nSYSTEM_PROMPT = "You are a professional English-Vietnamese translator."\nUSER_TEMPLATE = ("Translate the following English sentence into Vietnamese. "\n                 "Output only the Vietnamese translation, nothing else.\\n\\n{src}")\n\n\ndef build_messages(src, shots=()):\n    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]\n    for s, t in shots:\n        msgs.append({"role": "user", "content": USER_TEMPLATE.format(src=s)})\n        msgs.append({"role": "assistant", "content": t})\n    msgs.append({"role": "user", "content": USER_TEMPLATE.format(src=src)})\n    return msgs\n\n\ndef chat_prompt(tok, src, shots=()):\n    kw = {}\n    if "enable_thinking" in (getattr(tok, "chat_template", None) or ""):\n        kw["enable_thinking"] = False   # Qwen3: tắt chế độ "suy nghĩ"\n    s = tok.apply_chat_template(build_messages(src, shots), tokenize=False, add_generation_prompt=True, **kw)\n    if not isinstance(s, str):\n        raise RuntimeError("apply_chat_template(tokenize=False) không trả về chuỗi.")\n    return s\n\n\ndef chat_train_texts(tok, src, tgt):\n    """(prompt_text, target_text) sao cho prompt + target = hội thoại đầy đủ theo chat template."""\n    prompt = chat_prompt(tok, src)\n    full = tok.apply_chat_template(build_messages(src) + [{"role": "assistant", "content": tgt}],\n                                   tokenize=False, add_generation_prompt=False)\n    if not full.startswith(prompt):\n        raise RuntimeError("Chat template không nhất quán giữa prompt và hội thoại đầy đủ "\n                           "(thường gặp ở mô hình có chế độ suy nghĩ). Hãy dùng Qwen2.5-Instruct.")\n    target = full[len(prompt):]\n    eos = getattr(tok, "eos_token", None)\n    if eos and eos in target:   # bỏ phần sau token kết thúc (vd. "\\n" của ChatML) — không cần học\n        target = target[: target.index(eos) + len(eos)]\n    return prompt, target\n\n\n_PREFIX_RE = re.compile(r"^(vietnamese( translation)?|translation|bản dịch|dịch|tiếng việt)\\s*[:：]\\s*", re.I)\n\n\ndef clean_llm_output(text: str) -> str:\n    """Hậu xử lý đầu ra LLM: lấy dòng đầu không rỗng, bỏ tiền tố \'Translation:\' và dấu ngoặc kép bao ngoài."""\n    text = (text or "").replace("<think>", "").replace("</think>", "")\n    lines = [l.strip() for l in text.strip().splitlines() if l.strip()]\n    s = lines[0] if lines else ""\n    s = _PREFIX_RE.sub("", s)\n    if len(s) >= 2 and s[0] == s[-1] and s[0] in "\\"\'“”":\n        s = s[1:-1]\n    if len(s) >= 2 and s[0] == "“" and s[-1] == "”":\n        s = s[1:-1]\n    return normalize_text(s)\n\n\ndef load_llm(model_id, device, dtype):\n    from transformers import AutoModelForCausalLM\n    path = resolve_model_id(model_id)\n    tok = load_tokenizer(path)\n    if tok.pad_token is None:\n        tok.pad_token = tok.eos_token\n    if not getattr(tok, "chat_template", None):\n        raise RuntimeError(f"{model_id} không có chat template — hãy dùng bản *-Instruct.")\n    model = AutoModelForCausalLM.from_pretrained(path, **dtype_kw(dtype)).to(device)\n    if model.generation_config.pad_token_id is None:\n        model.generation_config.pad_token_id = tok.pad_token_id\n    return tok, model\n\n\ndef make_llm_features(tok, srcs, tgts, max_len=384):\n    feats, n_skip = [], 0\n    for s, t in zip(srcs, tgts):\n        p, tgt = chat_train_texts(tok, s, t)\n        p_ids = tok(p, add_special_tokens=False)["input_ids"]\n        t_ids = tok(tgt, add_special_tokens=False)["input_ids"]\n        if len(p_ids) + len(t_ids) > max_len:\n            n_skip += 1\n            continue\n        feats.append((p_ids, t_ids))\n    return feats, n_skip\n\n\ndef llm_collate(batch, pad_id):\n    from .data import IGNORE\n    seqs = [list(p) + list(t) for _, p, t in batch]\n    labs = [[IGNORE] * len(p) + list(t) for _, p, t in batch]\n    L = max(len(s) for s in seqs)\n    ids = torch.full((len(seqs), L), pad_id, dtype=torch.long)\n    lab = torch.full((len(seqs), L), IGNORE, dtype=torch.long)\n    mask = torch.zeros((len(seqs), L), dtype=torch.long)\n    for i, (s, l) in enumerate(zip(seqs, labs)):\n        ids[i, :len(s)] = torch.tensor(s)\n        lab[i, :len(l)] = torch.tensor(l)\n        mask[i, :len(s)] = 1\n    return {"input_ids": ids, "labels": lab, "attention_mask": mask}\n\n\ndef llm_loss(model, batch, cfg):\n    from .models import label_smoothed_ce\n    out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])\n    logits = out.logits[:, :-1]\n    labels = batch["labels"][:, 1:]\n    return label_smoothed_ce(logits, labels, cfg.label_smoothing)\n\n\n@torch.no_grad()\ndef llm_translate(model, tok, srcs, shots=(), batch_size=32, max_new_tokens=256, amp_dtype=None,\n                  progress=True, return_raw=False):\n    from tqdm.auto import tqdm\n    model.eval()\n    device = next(model.parameters()).device\n    old_side = tok.padding_side\n    tok.padding_side = "left"   # sinh văn bản theo batch với decoder-only: BẮT BUỘC đệm trái\n    prompts = [chat_prompt(tok, s, shots) for s in srcs]\n    batches = sorted_eval_batches([len(p) for p in prompts], batch_size)\n    out, raw = [None] * len(srcs), [None] * len(srcs)\n    eos = model.generation_config.eos_token_id\n    it = tqdm(batches, desc="LLM dịch", leave=False) if progress else batches\n    try:\n        for idx in it:\n            enc = tok([prompts[i] for i in idx], return_tensors="pt", padding=True, add_special_tokens=False)\n            enc = {k: enc[k].to(device) for k in ("input_ids", "attention_mask")}\n            with autocast_ctx(amp_dtype):\n                gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, num_beams=1,\n                                     use_cache=True, pad_token_id=tok.pad_token_id, eos_token_id=eos)\n            new = gen[:, enc["input_ids"].size(1):]\n            for i, h in zip(idx, decode_batch(tok, new)):\n                raw[i] = h\n                out[i] = clean_llm_output(h)\n    finally:\n        tok.padding_side = old_side\n    return (out, raw) if return_raw else out\n\n\ndef enable_grad_ckpt(peft_model):\n    """Gradient checkpointing: giảm bộ nhớ kích hoạt (activation) ~3-5 lần, đổi lại chậm hơn ~30%."""\n    base = peft_model.get_base_model() if hasattr(peft_model, "get_base_model") else peft_model\n    try:\n        base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n    except TypeError:\n        base.gradient_checkpointing_enable()\n    if hasattr(base, "enable_input_require_grads"):\n        base.enable_input_require_grads()\n    base.config.use_cache = False\n',
    'train.py': '"""Vòng lặp huấn luyện dùng chung cho MỌI mô hình (tự cài, HF seq2seq, LLM + LoRA).\n\nCác cơ chế phòng rủi ro:\n  * Checkpoint \'last.pt\' lưu định kỳ (model + optimizer + scheduler + scaler + RNG + vị trí batch).\n    Colab mất kết nối => chạy lại ô huấn luyện là tự động tiếp tục ĐÚNG chỗ cũ.\n  * Ghi file tạm rồi đổi tên => không bao giờ có checkpoint hỏng nửa chừng.\n  * Lưu \'best\' theo điểm dev (BLEU) + early stopping.\n  * Phát hiện loss NaN/inf: bỏ qua bước đó; quá nhiều lần liên tiếp => dừng và báo nguyên nhân.\n  * Bắt lỗi hết bộ nhớ GPU (OOM): bỏ qua batch; lặp lại nhiều lần => dừng và gợi ý giảm max_tokens.\n  * Giới hạn thời gian (time_budget_hours): dừng an toàn trước khi hết quota Colab.\n  * Ngắt tay (Interrupt): vẫn lưu checkpoint trước khi dừng.\n"""\nfrom __future__ import annotations\n\nimport csv\nimport math\nimport os\nimport random\nimport time\nfrom dataclasses import asdict, dataclass, field\nfrom pathlib import Path\nfrom typing import Callable\n\nimport numpy as np\nimport torch\n\nfrom .common import LIB_VERSION, autocast_ctx, fmt_seconds, now_str, save_json\n\n\n@dataclass\nclass TrainConfig:\n    epochs: int = 50\n    max_tokens: int = 8192            # số token tối đa / batch (trước tích luỹ gradient)\n    max_sentences: int | None = None  # giới hạn số câu / batch (tuỳ chọn)\n    grad_accum: int = 1\n    lr: float = 7e-4\n    warmup_steps: int = 4000\n    schedule: str = "inverse_sqrt"    # inverse_sqrt | linear | cosine | constant\n    min_lr_ratio: float = 0.0         # cho linear/cosine\n    weight_decay: float = 0.0001\n    betas: tuple = (0.9, 0.98)\n    adam_eps: float = 1e-8\n    clip_norm: float = 1.0\n    label_smoothing: float = 0.1\n    patience: int = 6                 # số lần đánh giá liên tiếp không cải thiện thì dừng\n    eval_every_steps: int | None = None  # None = đánh giá cuối mỗi epoch\n    save_every_steps: int = 1000      # lưu last.pt định kỳ giữa epoch\n    log_every_steps: int = 200\n    max_steps: int | None = None\n    time_budget_hours: float | None = None\n    seed: int = 42\n    amp: str = "auto"                 # auto | bf16 | fp16 | fp32\n    save_optimizer: bool = True\n\n    def to_dict(self):\n        d = asdict(self)\n        d["betas"] = list(self.betas)\n        return d\n\n\ndef make_scheduler(optimizer, cfg: TrainConfig, total_steps: int):\n    w = max(1, cfg.warmup_steps)\n\n    def f(step):\n        step = max(1, step)\n        if cfg.schedule == "inverse_sqrt":\n            return min(step / w, math.sqrt(w / step))\n        if step < w:\n            return step / w\n        if cfg.schedule == "constant":\n            return 1.0\n        prog = min(1.0, (step - w) / max(1, total_steps - w))\n        if cfg.schedule == "linear":\n            return max(cfg.min_lr_ratio, 1 - prog * (1 - cfg.min_lr_ratio))\n        if cfg.schedule == "cosine":\n            return cfg.min_lr_ratio + (1 - cfg.min_lr_ratio) * 0.5 * (1 + math.cos(math.pi * prog))\n        raise ValueError(cfg.schedule)\n\n    return torch.optim.lr_scheduler.LambdaLR(optimizer, f)\n\n\ndef _atomic_save(obj, path: Path):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    torch.save(obj, tmp)\n    os.replace(tmp, path)\n\n\ndef _rng_state():\n    st = {"py": random.getstate(), "np": np.random.get_state(), "torch": torch.get_rng_state()}\n    if torch.cuda.is_available():\n        st["cuda"] = torch.cuda.get_rng_state_all()\n    return st\n\n\ndef _set_rng_state(st):\n    try:\n        random.setstate(st["py"])\n        np.random.set_state(st["np"])\n        torch.set_rng_state(st["torch"])\n        if "cuda" in st and torch.cuda.is_available():\n            torch.cuda.set_rng_state_all(st["cuda"])\n    except Exception as e:  # noqa: BLE001\n        print(f"⚠️  Không khôi phục được trạng thái ngẫu nhiên ({e}) — vẫn tiếp tục.")\n\n\nclass TooManyFailures(RuntimeError):\n    pass\n\n\nclass Trainer:\n    """\n    loss_fn(model, batch, cfg) -> (loss, nll_mean, n_tokens)   [batch đã ở trên device]\n    eval_fn(model) -> dict, bắt buộc có khoá \'score\' (càng cao càng tốt, ví dụ BLEU dev)\n    save_best_fn(model, dir) -> None : tuỳ chọn, cách lưu mô hình tốt nhất (mặc định state_dict)\n    """\n\n    def __init__(self, model, dataset, collate_fn, lengths, cfg: TrainConfig, ckpt_dir,\n                 loss_fn: Callable, eval_fn: Callable | None = None, device=None,\n                 save_best_fn: Callable | None = None, run_name: str = "run",\n                 extra_meta: dict | None = None, params=None, last_dir=None):\n        self.model = model\n        self.dataset = dataset\n        self.collate_fn = collate_fn\n        self.lengths = lengths\n        self.cfg = cfg\n        self.ckpt_dir = Path(ckpt_dir)\n        self.ckpt_dir.mkdir(parents=True, exist_ok=True)\n        # last.pt (kèm trạng thái optimizer, có thể nặng vài GB) có thể đặt ở ổ máy ảo thay vì Drive\n        self.last_dir = Path(last_dir) if last_dir else self.ckpt_dir\n        self.last_dir.mkdir(parents=True, exist_ok=True)\n        self.loss_fn = loss_fn\n        self.eval_fn = eval_fn\n        self.device = device or next(model.parameters()).device\n        self.save_best_fn = save_best_fn\n        self.run_name = run_name\n        self.extra_meta = extra_meta or {}\n        from .common import pick_amp_dtype\n        self.amp_dtype = pick_amp_dtype(cfg.amp) if self.device.type == "cuda" else None\n        self.use_scaler = self.amp_dtype == torch.float16\n        self.scaler = torch.amp.GradScaler("cuda", enabled=self.use_scaler)\n\n        from .data import TokenBatchSampler\n        self.sampler = TokenBatchSampler(lengths, cfg.max_tokens, shuffle=True, seed=cfg.seed,\n                                         max_sentences=cfg.max_sentences)\n        self.steps_per_epoch = math.ceil(len(self.sampler.batches(0)) / cfg.grad_accum)\n        self.total_steps = cfg.max_steps or self.steps_per_epoch * cfg.epochs\n\n        params = [p for p in (params if params is not None else model.parameters()) if p.requires_grad]\n        decay = [p for p in params if p.ndim >= 2]\n        no_decay = [p for p in params if p.ndim < 2]\n        self.optimizer = torch.optim.AdamW(\n            [{"params": decay, "weight_decay": cfg.weight_decay}, {"params": no_decay, "weight_decay": 0.0}],\n            lr=cfg.lr, betas=tuple(cfg.betas), eps=cfg.adam_eps)\n        self.scheduler = make_scheduler(self.optimizer, cfg, self.total_steps)\n\n        self.state = {"epoch": 0, "batch_in_epoch": 0, "global_step": 0, "best_score": -float("inf"),\n                      "best_step": 0, "bad_evals": 0, "train_seconds": 0.0, "finished": False}\n        self.history = []\n\n    # ---------------------------------------------------------------- checkpoint\n    @property\n    def last_path(self):\n        return self.last_dir / "last.pt"\n\n    def _trainable_state(self):\n        # Với LoRA chỉ lưu tham số được huấn luyện (vài chục MB thay vì vài GB)\n        return {k: v.detach().cpu() for k, v in self.model.state_dict().items()\n                if k in self._trainable_names}\n\n    @property\n    def _trainable_names(self):\n        if not hasattr(self, "_tn"):\n            names = {n for n, p in self.model.named_parameters() if p.requires_grad}\n            all_params = {n for n, _ in self.model.named_parameters()}\n            # Nếu train toàn bộ thì lưu cả state_dict (gồm buffer)\n            self._tn = None if names == all_params else names\n        return self._tn\n\n    def save_last(self):\n        sd = self.model.state_dict() if self._trainable_names is None else self._trainable_state()\n        ck = {"model": sd, "state": self.state, "history": self.history, "rng": _rng_state(),\n              "cfg": self.cfg.to_dict(), "lib_version": LIB_VERSION, "saved_at": now_str(),\n              "scheduler": self.scheduler.state_dict(), "scaler": self.scaler.state_dict()}\n        if self.cfg.save_optimizer:\n            ck["optimizer"] = self.optimizer.state_dict()\n        _atomic_save(ck, self.last_path)\n\n    def try_resume(self) -> bool:\n        if not self.last_path.exists():\n            return False\n        ck = torch.load(self.last_path, map_location="cpu", weights_only=False)\n        old = ck.get("cfg", {})\n        important = ["lr", "max_tokens", "grad_accum", "schedule", "warmup_steps", "seed", "epochs"]\n        diffs = {k: (old.get(k), self.cfg.to_dict().get(k)) for k in important if old.get(k) != self.cfg.to_dict().get(k)}\n        if diffs:\n            print(f"⚠️  Cấu hình khác với checkpoint cũ: {diffs}. Tiếp tục với cấu hình MỚI "\n                  "(nếu muốn huấn luyện lại từ đầu: xoá thư mục checkpoint của run này).")\n        try:\n            missing, unexpected = self.model.load_state_dict(ck["model"], strict=self._trainable_names is None)\n        except RuntimeError as e:\n            raise RuntimeError(\n                f"Checkpoint {self.last_path} không khớp với mô hình hiện tại (thường do đã đổi mô hình/cấu hình "\n                f"nhưng còn checkpoint cũ). Xoá file đó (và thư mục {self.ckpt_dir}) rồi chạy lại.\\n{str(e)[:300]}") from None\n        if self._trainable_names is not None and unexpected:\n            raise RuntimeError(f"Checkpoint có tham số lạ: {unexpected[:5]}")\n        if "optimizer" in ck:\n            self.optimizer.load_state_dict(ck["optimizer"])\n        else:\n            print("ℹ️  Checkpoint không có trạng thái optimizer — khởi tạo lại optimizer.")\n        self.scheduler.load_state_dict(ck["scheduler"])\n        if self.use_scaler and ck.get("scaler"):\n            self.scaler.load_state_dict(ck["scaler"])\n        self.state.update(ck["state"])\n        self.history = ck.get("history", [])\n        _set_rng_state(ck.get("rng", {}))\n        print(f"🔁 Tiếp tục từ checkpoint ({ck.get(\'saved_at\')}): epoch {self.state[\'epoch\'] + 1}, "\n              f"batch {self.state[\'batch_in_epoch\']}, step {self.state[\'global_step\']}, "\n              f"best dev = {self.state[\'best_score\']:.2f}")\n        return True\n\n    def _save_best(self):\n        if self.save_best_fn is not None:\n            self.save_best_fn(self.model, self.ckpt_dir / "best")\n        else:\n            _atomic_save({"model": self.model.state_dict(), "step": self.state["global_step"],\n                          "score": self.state["best_score"], "meta": self.extra_meta}, self.ckpt_dir / "best.pt")\n\n    def _log_history(self, row):\n        self.history.append(row)\n        path = self.ckpt_dir / "history.csv"\n        keys = sorted({k for r in self.history for k in r})\n        tmp = path.with_suffix(".csv.tmp")\n        with open(tmp, "w", newline="", encoding="utf-8") as f:\n            w = csv.DictWriter(f, fieldnames=keys)\n            w.writeheader()\n            w.writerows(self.history)\n        os.replace(tmp, path)\n\n    # ---------------------------------------------------------------- evaluate\n    def evaluate(self, tag=""):\n        row = {"step": self.state["global_step"], "epoch": self.state["epoch"],\n               "lr": self.optimizer.param_groups[0]["lr"], "time": now_str()}\n        self.state["last_eval_step"] = self.state["global_step"]\n        if self.eval_fn is not None:\n            self.model.eval()\n            res = self.eval_fn(self.model)\n            self.model.train()\n            row.update(res)\n            score = res["score"]\n            improved = score > self.state["best_score"] + 1e-6\n            if improved:\n                self.state["best_score"], self.state["best_step"], self.state["bad_evals"] = score, self.state["global_step"], 0\n                self._save_best()\n            else:\n                self.state["bad_evals"] += 1\n            row["best"] = improved\n            extra = " ".join(f"{k}={v:.3f}" if isinstance(v, float) else f"{k}={v}" for k, v in res.items() if k != "score")\n            if improved:\n                note = "⭐ tốt nhất mới"\n            else:\n                note = "(không cải thiện %d/%d)" % (self.state["bad_evals"], self.cfg.patience)\n            print(f"   📊 {tag} step {self.state[\'global_step\']}: dev score={score:.2f} {extra} {note}")\n        self._log_history(row)\n\n    # ---------------------------------------------------------------- train\n    def train(self):\n        from tqdm.auto import tqdm\n        cfg = self.cfg\n        self.try_resume()\n        if self.state["finished"]:\n            reason = self.state.get("stop_reason")\n            if reason == "đủ số epoch" and self.state["epoch"] < cfg.epochs:\n                print(f"➕ Tăng số epoch lên {cfg.epochs}: huấn luyện tiếp.")\n                self.state["finished"] = False\n            elif reason == "đạt max_steps" and (cfg.max_steps is None or self.state["global_step"] < cfg.max_steps):\n                print("➕ Tăng/bỏ giới hạn max_steps: huấn luyện tiếp.")\n                self.state["finished"] = False\n        if self.state["finished"]:\n            print("✔ Run này đã huấn luyện xong trước đó. (Xoá thư mục checkpoint của run nếu muốn huấn luyện lại.)")\n            return self.history\n        if cfg.max_steps and self.state["global_step"] >= cfg.max_steps:\n            print(f"✔ Đã đạt max_steps={cfg.max_steps}.")\n            return self.history\n        self.model.train()\n        t_start = time.time()\n        sec_before = self.state["train_seconds"]\n        n_bad_loss = n_oom = 0\n        stop_reason = None\n        loss_acc = nll_acc = tok_acc = 0.0\n        since_log = 0\n\n        try:\n            for epoch in range(self.state["epoch"], cfg.epochs):\n                self.state["epoch"] = epoch\n                batches = self.sampler.batches(epoch)\n                start_b = self.state["batch_in_epoch"]\n                loader = torch.utils.data.DataLoader(self.dataset, batch_sampler=batches[start_b:],\n                                                     collate_fn=self.collate_fn, num_workers=2,\n                                                     persistent_workers=False, pin_memory=self.device.type == "cuda",\n                                                     # generator riêng => DataLoader không "ăn" RNG toàn cục,\n                                                     # dropout sau khi resume giống hệt khi chạy liền mạch\n                                                     generator=torch.Generator().manual_seed(cfg.seed + epoch))\n                pbar = tqdm(total=len(batches), initial=start_b, desc=f"{self.run_name} epoch {epoch + 1}/{cfg.epochs}",\n                            leave=False)\n                self.optimizer.zero_grad(set_to_none=True)\n                micro = 0\n                for bi, batch in enumerate(loader, start=start_b):\n                    batch = {k: v.to(self.device, non_blocking=True) for k, v in batch.items()}\n                    try:\n                        with autocast_ctx(self.amp_dtype):\n                            loss, nll, ntok = self.loss_fn(self.model, batch, cfg)\n                        if not torch.isfinite(loss):\n                            n_bad_loss += 1\n                            self.optimizer.zero_grad(set_to_none=True)\n                            micro = 0\n                            if n_bad_loss >= 20:\n                                raise TooManyFailures(\n                                    "Loss bị NaN/inf 20 lần liên tiếp. Thử: giảm lr, đặt amp=\'fp32\' hoặc \'bf16\'.")\n                            continue\n                        n_bad_loss = 0\n                        self.scaler.scale(loss / cfg.grad_accum).backward()\n                    except torch.cuda.OutOfMemoryError:\n                        n_oom += 1\n                        self.optimizer.zero_grad(set_to_none=True)\n                        micro = 0\n                        del batch\n                        torch.cuda.empty_cache()\n                        print(f"⚠️  Hết bộ nhớ GPU ở batch {bi} (lần {n_oom}) — bỏ qua batch này.")\n                        if n_oom >= 10:\n                            raise TooManyFailures("Hết bộ nhớ GPU quá nhiều lần. Giảm max_tokens (vd. chia đôi) "\n                                                  "và tăng grad_accum tương ứng.")\n                        continue\n                    loss_acc += float(loss.detach()) * ntok\n                    nll_acc += float(nll) * ntok\n                    tok_acc += ntok\n                    micro += 1\n                    pbar.update(1)\n                    self.state["batch_in_epoch"] = bi + 1\n                    if micro < cfg.grad_accum:\n                        continue\n                    micro = 0\n                    self.scaler.unscale_(self.optimizer)\n                    gnorm = torch.nn.utils.clip_grad_norm_(\n                        [p for g in self.optimizer.param_groups for p in g["params"]], cfg.clip_norm)\n                    self.scaler.step(self.optimizer)\n                    self.scaler.update()\n                    self.optimizer.zero_grad(set_to_none=True)\n                    self.scheduler.step()\n                    self.state["global_step"] += 1\n                    since_log += 1\n                    gs = self.state["global_step"]\n\n                    if gs % cfg.log_every_steps == 0 and tok_acc > 0:\n                        tl = loss_acc / tok_acc\n                        ppl = math.exp(min(20, nll_acc / tok_acc))\n                        pbar.set_postfix(loss=f"{tl:.3f}", ppl=f"{ppl:.1f}", lr=f"{self.optimizer.param_groups[0][\'lr\']:.2e}")\n                        self._log_history({"step": gs, "epoch": epoch, "train_loss": tl, "train_ppl": ppl,\n                                           "grad_norm": float(gnorm), "lr": self.optimizer.param_groups[0]["lr"],\n                                           "time": now_str()})\n                        loss_acc = nll_acc = tok_acc = 0.0\n                    if cfg.eval_every_steps and gs % cfg.eval_every_steps == 0:\n                        self.evaluate(tag=f"epoch {epoch + 1}")\n                        if self.state["bad_evals"] >= cfg.patience:\n                            stop_reason = "early stopping"\n                    if gs % cfg.save_every_steps == 0:\n                        self.state["train_seconds"] = sec_before + time.time() - t_start\n                        self.save_last()\n                    if cfg.max_steps and gs >= cfg.max_steps:\n                        stop_reason = stop_reason or "đạt max_steps"\n                    if cfg.time_budget_hours and (time.time() - t_start) / 3600 > cfg.time_budget_hours:\n                        stop_reason = stop_reason or "hết thời gian cho phép (time_budget_hours)"\n                    if stop_reason:\n                        break\n                pbar.close()\n                if stop_reason:\n                    break\n                # cuối epoch\n                self.state["epoch"] = epoch + 1\n                self.state["batch_in_epoch"] = 0\n                if not cfg.eval_every_steps:\n                    self.evaluate(tag=f"epoch {epoch + 1}/{cfg.epochs}")\n                    if self.state["bad_evals"] >= cfg.patience:\n                        stop_reason = "early stopping"\n                self.state["train_seconds"] = sec_before + time.time() - t_start\n                self.save_last()\n                if stop_reason:\n                    break\n        except KeyboardInterrupt:\n            self.state["train_seconds"] = sec_before + time.time() - t_start\n            self.save_last()\n            print("⏸ Đã dừng tay và lưu checkpoint. Chạy lại ô này để tiếp tục.")\n            raise\n\n        self.state["train_seconds"] = sec_before + time.time() - t_start\n        # Chỉ coi là "xong hẳn" khi dừng sớm hoặc đủ epoch; dừng vì max_steps/hết giờ thì lần sau chạy tiếp được\n        # Đánh giá lần cuối nếu bước hiện tại chưa được đánh giá (vd. dừng vì max_steps giữa epoch)\n        if (self.eval_fn is not None and stop_reason != "hết thời gian cho phép (time_budget_hours)"\n                and self.state.get("last_eval_step") != self.state["global_step"]):\n            self.evaluate(tag="cuối")\n        if stop_reason in (None, "early stopping", "đạt max_steps"):\n            self.state["finished"] = True\n        self.state["stop_reason"] = stop_reason or "đủ số epoch"\n        self.save_last()\n        save_json({"run": self.run_name, **self.state,\n                   "train_time": fmt_seconds(self.state["train_seconds"]), "cfg": self.cfg.to_dict(),\n                   **self.extra_meta}, self.ckpt_dir / "train_summary.json")\n        print(f"🏁 {self.run_name}: kết thúc ({stop_reason or \'đủ số epoch\'}), "\n              f"best dev = {self.state[\'best_score\']:.2f} tại step {self.state[\'best_step\']}, "\n              f"thời gian train = {fmt_seconds(self.state[\'train_seconds\'])}")\n        return self.history\n',
}
for _name, _src in _FILES.items():
    _p = os.path.join(PROJECT_DIR, 'envi_lib', _name)
    _old = open(_p, encoding='utf-8').read() if os.path.exists(_p) else None
    if _old != _src:
        with open(_p, 'w', encoding='utf-8') as _f:
            _f.write(_src)
        print('✔ Đã cập nhật envi_lib/' + _name)
    sys.modules.pop('envi_lib.' + _name[:-3], None)   # bắt Python nạp lại bản mới
print('Thư viện đã là bản mới nhất.')

Thư viện đã là bản mới nhất.


In [4]:
%pip install -q sacrebleu sacremoses sentencepiece peft
import re, time, random, torch, pandas as pd, transformers, peft
from functools import partial
from envi_lib import data as D, metrics as M, hf_utils as H
from envi_lib.train import TrainConfig, Trainer
print("transformers", transformers.__version__, "| peft", peft.__version__, "| GPU:", gpu_info())
if DEVICE.type != "cuda" and not DRY_RUN:
    raise RuntimeError("Cần GPU: Runtime → Change runtime type → GPU")
mid = DRY_MODEL_ID if DRY_RUN else MODEL_ID
TAG = re.sub(r"[^a-z0-9.]+", "-", mid.split("/")[-1].lower()).replace("-instruct", "")
INFER_DTYPE = torch.bfloat16 if bf16_supported() else (torch.float16 if DEVICE.type == "cuda" else torch.float32)
TRAIN_DTYPE = torch.bfloat16 if bf16_supported() else torch.float32   # T4: trọng số fp32 + autocast fp16
AMP = pick_amp_dtype("auto")
gpu_gb = gpu_info().get("total_gb", 0)
data = D.load_all_clean(paths, DRY_RUN)
train_src, train_tgt = data["train"]; dev_src, dev_ref = data["dev"]; test_src, test_ref = data["test"]
dev_n = 30 if DRY_RUN else DEV_EVAL_SIZE
print(f"Mô hình: {mid} (tên hệ thống: {TAG}_*) | dtype suy luận {INFER_DTYPE} | huấn luyện {TRAIN_DTYPE}")

ℹ️  Phát hiện torchao bản cũ của Colab — đã báo peft bỏ qua torchao (không dùng tới trong đồ án).
transformers 5.20.1 | peft 0.21.2 | GPU: {'cuda': True, 'torch': '2.11.0+cu130', 'name': 'NVIDIA L4', 'total_gb': 22.0, 'bf16': True}
Mô hình: Qwen/Qwen2.5-1.5B-Instruct (tên hệ thống: qwen2.5-1.5b_*) | dtype suy luận torch.bfloat16 | huấn luyện torch.bfloat16


In [5]:
#@title 🧪 Preflight: chat template, prompt mẫu, dịch thử
tok, model = H.load_llm(mid, DEVICE, INFER_DTYPE)
rng = random.Random(42)
pool = [i for i, s in enumerate(train_src) if 8 <= len(s.split()) <= 20]
SHOTS = [(train_src[i], train_tgt[i]) for i in rng.sample(pool, N_SHOTS)]
print("PROMPT ZERO-SHOT:\n" + H.chat_prompt(tok, test_src[0]))
p, t = H.chat_train_texts(tok, train_src[0], train_tgt[0])
print("MỤC TIÊU HUẤN LUYỆN (phần được tính loss):", repr(t))
out, raw = H.llm_translate(model, tok, test_src[:3], batch_size=3, max_new_tokens=128, amp_dtype=None, progress=False, return_raw=True)
for s, r, o in zip(test_src[:3], raw, out):
    print(f"\nEN : {s}\nRAW: {r!r}\nOUT: {o}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

PROMPT ZERO-SHOT:
<|im_start|>system
You are a professional English-Vietnamese translator.<|im_end|>
<|im_start|>user
Translate the following English sentence into Vietnamese. Output only the Vietnamese translation, nothing else.

When I was little, I thought my country was the best on the planet, and I grew up singing a song called "Nothing To Envy."<|im_end|>
<|im_start|>assistant

MỤC TIÊU HUẤN LUYỆN (phần được tính loss): 'Khoa học đằng sau một tiêu đề về khí hậu<|im_end|>'

EN : When I was little, I thought my country was the best on the planet, and I grew up singing a song called "Nothing To Envy."
RAW: 'Khi tôi còn nhỏ, tôi nghĩ quốc gia của mình là tốt nhất trên hành tinh, và tôi lớn lên hát bài hát "Không có gì để羡慕".'
OUT: Khi tôi còn nhỏ, tôi nghĩ quốc gia của mình là tốt nhất trên hành tinh, và tôi lớn lên hát bài hát "Không có gì để羡慕".

EN : And I was very proud.
RAW: 'Tôi rất tự hào.'
OUT: Tôi rất tự hào.

EN : In school, we spent a lot of time studying the history of Ki

In [6]:
#@title 🚀 Zero-shot & few-shot
def run_prompting(system, shots, bs):
    fp = {"model_id": mid, "shots": len(shots), "prompt": H.USER_TEMPLATE}
    if prediction_is_current(paths, system, fp, len(test_ref)):
        return
    reset_peak_mem(); t0 = time.time()
    hyps, raw = H.llm_translate(model, tok, test_src, shots=shots, batch_size=bs, max_new_tokens=256, return_raw=True)
    dt = time.time() - t0
    save_predictions(paths, system, hyps, expected_n=len(test_ref), meta={
        "fingerprint": fp, "architecture": "Decoder-only (LLM)", "model_id": mid,
        "training": f"Pretrained ({'zero-shot' if not shots else str(len(shots)) + '-shot'})",
        "params": count_params(model), "decode_time_s": dt, "beam": 1, "peak_mem_gb": peak_gpu_mem_gb()})
    save_json([{"src": s, "raw": r, "clean": h} for s, r, h in zip(test_src[:100], raw[:100], hyps[:100])],
              paths.results / f"{system}_raw_samples.json")
    print(f"   {system}: test BLEU = {M.quick_bleu(hyps, test_ref):.2f} [{dt:.0f}s]")

if RUN_ZERO_SHOT:
    run_prompting(f"{TAG}_zeroshot", (), GEN_BATCH)
if RUN_FEW_SHOT:
    run_prompting(f"{TAG}_{N_SHOTS}shot", SHOTS, max(1, GEN_BATCH // 2))
del model; free_memory()

✔ qwen2.5-1.5b_zeroshot: đã có bản dịch từ đúng checkpoint này — bỏ qua (xoá file trong predictions/ nếu muốn dịch lại).
✔ qwen2.5-1.5b_5shot: đã có bản dịch từ đúng checkpoint này — bỏ qua (xoá file trong predictions/ nếu muốn dịch lại).


In [7]:
#@title 🚀 Fine-tune LoRA
if RUN_LORA:
    from peft import LoraConfig, get_peft_model, PeftModel
    tok, base = H.load_llm(mid, DEVICE, TRAIN_DTYPE)
    model = get_peft_model(base, LoraConfig(task_type="CAUSAL_LM", **LORA))
    use_gc = DEVICE.type == "cuda" and gpu_gb < 30
    if use_gc:
        H.enable_grad_ckpt(model)
    n_tr, n_all = count_params(model, trainable_only=True), count_params(model)
    print(f"Tham số huấn luyện: {n_tr / 1e6:.1f}M / {n_all / 1e6:.0f}M ({100 * n_tr / n_all:.2f}%) | gradient checkpointing: {use_gc}")
    src_tr, tgt_tr = train_src, train_tgt
    if TRAIN_SUBSET and TRAIN_SUBSET < len(src_tr):
        idx = random.Random(0).sample(range(len(src_tr)), TRAIN_SUBSET)
        src_tr, tgt_tr = [src_tr[i] for i in idx], [tgt_tr[i] for i in idx]
    feats, n_skip = H.make_llm_features(tok, src_tr, tgt_tr, max_len=384)
    print(f"{len(feats):,} mẫu huấn luyện (bỏ {n_skip} mẫu quá dài)")
    ft = dict(FT)
    if DRY_RUN:
        ft.update(epochs=1, warmup_steps=2, max_tokens=2048, eval_every_steps=10, save_every_steps=10)
    if gpu_gb and gpu_gb < 30 and not DRY_RUN:
        ft.update(max_tokens=ft["max_tokens"] // 2, grad_accum=ft["grad_accum"] * 2)
    tcfg = TrainConfig(**ft, label_smoothing=0.0, weight_decay=0.0, betas=(0.9, 0.999), clip_norm=1.0,
                       log_every_steps=5 if DRY_RUN else 50, max_steps=20 if DRY_RUN else None,
                       time_budget_hours=TIME_BUDGET_HOURS)
    ck = paths.checkpoints / f"{TAG}_lora"

    def eval_fn(m):
        hy = H.llm_translate(m, tok, dev_src[:dev_n], batch_size=GEN_BATCH, max_new_tokens=256, amp_dtype=AMP, progress=False)
        return {"score": M.quick_bleu(hy, dev_ref[:dev_n])}

    trainer = Trainer(model, H.FeatDataset(feats), partial(H.llm_collate, pad_id=tok.pad_token_id),
                      [len(a) + len(b) for a, b in feats], tcfg, ck, H.llm_loss, eval_fn, device=DEVICE,
                      save_best_fn=lambda m, d: m.save_pretrained(d), run_name=f"{TAG}_lora",
                      extra_meta={"model_id": mid, "lora": LORA})
    trainer.train()
    finished, best_step, train_s = trainer.state["finished"], trainer.state["best_step"], trainer.state["train_seconds"]
    del trainer, model, base; free_memory()
    if finished:
        system = f"{TAG}_lora"
        fp = {"model_id": mid, "ckpt_step": best_step, "lora": LORA}
        if not prediction_is_current(paths, system, fp, len(test_ref)):
            tok, base = H.load_llm(mid, DEVICE, INFER_DTYPE)
            model = PeftModel.from_pretrained(base, str(ck / "best")).merge_and_unload()   # gộp LoRA => dịch nhanh hơn
            reset_peak_mem(); t0 = time.time()
            hyps = H.llm_translate(model, tok, test_src, batch_size=GEN_BATCH, max_new_tokens=256)
            dt = time.time() - t0
            save_predictions(paths, system, hyps, expected_n=len(test_ref), meta={
                "fingerprint": fp, "architecture": "Decoder-only (LLM)", "model_id": mid,
                "training": "Pretrained + LoRA fine-tune", "params": count_params(model), "trainable_params": n_tr,
                "train_time_s": train_s, "decode_time_s": dt, "beam": 1, "peak_mem_gb": peak_gpu_mem_gb()})
            print(f"   {system}: test BLEU = {M.quick_bleu(hyps, test_ref):.2f}")
            del model, base; free_memory()
    else:
        print("⏸ Chưa xong (giới hạn thời gian) — chạy lại ô này để tiếp tục.")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Tham số huấn luyện: 18.5M / 1562M (1.18%) | gradient checkpointing: True
130,398 mẫu huấn luyện (bỏ 2 mẫu quá dài)
🔁 Tiếp tục từ checkpoint (2026-10-23 01:03:52): epoch 1, batch 1665, step 832, best dev = 29.59


qwen2.5-1.5b_lora epoch 1/1:  57%|#####6    | 1665/2925 [00:00<?, ?it/s]

   📊 epoch 1 step 1000: dev score=30.88  ⭐ tốt nhất mới
   📊 cuối step 1462: dev score=30.80  (không cải thiện 1/4)
🏁 qwen2.5-1.5b_lora: kết thúc (đủ số epoch), best dev = 30.88 tại step 1000, thời gian train = 1h32m40s


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

LLM dịch:   0%|          | 0/40 [00:00<?, ?it/s]

✅ Đã lưu 1268 bản dịch: /content/drive/MyDrive/NLP_Final_EnVi/predictions/qwen2.5-1.5b_lora.vi.txt
   qwen2.5-1.5b_lora: test BLEU = 31.35


In [8]:
#@title 📋 Tóm tắt nhanh
rows = []
for f in sorted(paths.predictions.glob(f"{TAG}_*.vi.txt")):
    system = f.name.replace(".vi.txt", ""); meta = load_json(paths.predictions / f"{system}.meta.json")
    hy = read_lines(f)
    rows.append({"hệ thống": system, "chế độ": meta["training"], "BLEU test": round(M.quick_bleu(hy, test_ref), 2),
                 "câu rỗng": meta["n_empty"], "thời gian dịch (s)": round(meta["decode_time_s"], 1)})
pd.DataFrame(rows)

,hệ thống,chế độ,BLEU test,câu rỗng,thời gian dịch (s)
0,qwen2.5-1.5b_5shot,Pretrained (5-shot),23.62,0,150.7
1,qwen2.5-1.5b_lora,Pretrained + LoRA fine-tune,31.35,0,72.5
2,qwen2.5-1.5b_zeroshot,Pretrained (zero-shot),23.92,0,69.2
